##### 카테고리

In [54]:
# from pathlib import Path
# import zipfile
# import shutil

# base_dir = Path(r"E:\all_pass_카테고리_붙인_데이터")
# output_dir = base_dir / "category"
# output_dir.mkdir(exist_ok=True)

# for zip_path in base_dir.glob("category_*.zip"):
#     press = zip_path.stem.replace("category_", "")
#     target = f"category/outputs/corpus_3_categorized/{press}.json"

#     with zipfile.ZipFile(zip_path) as zf:
#         if target not in zf.namelist():
#             print(f"제외: {press}")
#             continue

#         with zf.open(target) as src, open(output_dir / f"{press}.json", "wb") as dst:
#             shutil.copyfileobj(src, dst)

#     print(f"완료: {press}.json")

##### 개체명 목록 생성

In [2]:
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')
from src.postgresql.query_reviews import get_connection

- review_method = '일괄처리', review_status = '완료'대상 중 언론사별 기사 추출

In [2]:
query = """
WITH article_ids AS (
SELECT 
DISTINCT press, 
doc_id 
FROM public.ner_human_review 
WHERE TRIM(review_method) = %s
  AND TRIM(review_status) = %s
), 
ranked AS (
SELECT 
press, 
doc_id, 
ROW_NUMBER() OVER (PARTITION BY press ORDER BY doc_id) AS article_rank 
FROM article_ids
) 
SELECT 
press,
doc_id 
FROM ranked 
WHERE 1=1
--AND article_rank <= 10 
ORDER BY press, article_rank
"""

with get_connection() as connection:
    rows = connection.execute(query, ('일괄처리','완료')).fetchall()

document_ids = [(row['press'], row['doc_id']) if hasattr(row, 'keys') else (row[0], row[1]) for row in rows]
len(document_ids)

12988

- 원본 데이터 가져오기 (ner_documents)

In [3]:
import pandas as pd

doc_id_values = [item[1] for item in document_ids]

query = """
SELECT
    split_part(doc_id, '.', 1) AS file_id,
    doc_id,
    paragraph_id,
    sentence_id,
    form,
    topic    
FROM public.ner_documents
WHERE 1=1
AND doc_id = ANY(%s)
AND topic = '사회'
ORDER BY doc_id, paragraph_id, sentence_id
"""

with get_connection() as connection:
    source_rows = connection.execute(query, (doc_id_values,)).fetchall()

In [4]:
source_df = pd.DataFrame(source_rows, columns=['file_id', 'doc_id', 'paragraph_id', 'sentence_id', 'topic', 'form'])

# 언론사별 10건
# source_df_by_file = source_df.groupby('file_id', group_keys=False).head(10).reset_index(drop=True)
# source_df_by_file

# 언론사별 고유 기사 10건
selected_docs = (
    source_df[['file_id', 'doc_id']]
    .drop_duplicates()
    .groupby('file_id', group_keys=False)
    .head(10)
)

# 선정된 기사의 전체 문장 추출
source_df_by_file = source_df.merge(selected_docs, on=['file_id', 'doc_id'], how='inner').reset_index(drop=True)
source_df_by_file

,file_id,doc_id,paragraph_id,sentence_id,topic,form
0,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.1,NIRW2600000001.10049.1.1,사회,"‘<서부지법: OG> 폭동’ 대응 실패 <마포경찰서: OG>장…<경찰청: OG>, ..."
1,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.2,NIRW2600000001.10049.2.1,사회,"<경찰청: OG>, <마포서: OG> <고석길: PS> 서장 등 직권경고 방침"
2,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.3,NIRW2600000001.10049.3.1,사회,<경찰청: OG>이 <서울서부지법: OG> 폭동 당시 대응에 실패한 <서울: LC>...
3,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.4,NIRW2600000001.10049.4.1,사회,<경찰청: OG>은 <12일: DT> <마포서: OG> <고석길: PS> 서장과 경...
4,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.4,NIRW2600000001.10049.4.2,사회,당사자들에게는 아직 내용이 전달되지 않은 상태다.
...,...,...,...,...,...,...
2635,NZRW2600000003,NZRW2600000003.111801,NZRW2600000003.111801.6,NZRW2600000003.111801.6.2,사회,부상자 16명 중 10명은 병원에 입원했다.
2636,NZRW2600000003,NZRW2600000003.111801,NZRW2600000003.111801.7,NZRW2600000003.111801.7.1,사회,공습은 자정께 시작돼 약 10시간 지속된 것으로 알려졌다.
2637,NZRW2600000003,NZRW2600000003.111801,NZRW2600000003.111801.7,NZRW2600000003.111801.7.2,사회,"아파트, 주유소 등 <키이우: LC> 곳곳에서 화재가 발생했고, <체르니히우: LC..."
2638,NZRW2600000003,NZRW2600000003.111801,NZRW2600000003.111801.8,NZRW2600000003.111801.8.1,사회,<볼로디미르 젤렌스키: PS> 대통령이 소셜미디어 <엑스: AF>(<X: AF>·구...


In [5]:
review_query = """
SELECT doc_id, sentence_id, modified_form
FROM public.ner_human_review
WHERE review_method = %s
  AND review_status = %s
  AND doc_id = ANY(%s)
"""

with get_connection() as connection:
    review_rows = connection.execute(review_query, ('일괄처리', '완료', doc_id_values)).fetchall()

review_df = pd.DataFrame(review_rows, columns=['doc_id', 'sentence_id', 'modified_form'])
final_df = source_df_by_file.merge(
    review_df,
    on=['doc_id', 'sentence_id'],
    how='left',
)
final_df['final_form'] = final_df['modified_form'].where(
    final_df['modified_form'].notna() & final_df['modified_form'].ne(''),
    final_df['form'],
)

- final_form 기준 개체명, 분류 컬럼 생성

In [6]:
import re
import pandas as pd

final_df_ = final_df[['file_id', 'doc_id', 'paragraph_id', 'sentence_id', 'final_form']].copy()

entity_pattern = r'<\s*([^<>:]+?)\s*:\s*([^<>]+?)\s*>'

base_df = final_df_.reset_index(drop=True)
base_df['_row_id'] = base_df.index

entity_df = (
    base_df.set_index('_row_id')['final_form']
    .fillna('')
    .str.extractall(entity_pattern)
    .rename(columns={0: '개체명', 1: '분류'})
    .reset_index()
    .merge(base_df, on='_row_id', how='left')
    .drop(columns=['_row_id', 'match'])
)

entity_df = entity_df[
    ['file_id', 'doc_id', 'paragraph_id', 'sentence_id',
     'final_form', '개체명', '분류']
]

In [ ]:
# entity_df[entity_df['sentence_id'] == 'NIRW2600000001.10049.1.1']
# final_df[final_df['sentence_id'] == 'NIRW2600000001.10049.1.1']['final_form'][0]

- 컬럼명: 개체명, 분류, 출현빈도
- 출현빈도 집계 기준: 현재 테이블 기준으로 전체 데이터에서 [개체명, 분류]가 언급된 총 갯수
- 저장 파일명: 개체명 목록_빈도.xlsx

In [10]:
from pathlib import Path
import pandas as pd
OUTPUT_DIR = Path(r"C:\work\ner\outputs\final")
press_map = {doc_id: press_id for press_id, doc_id in document_ids}

entity_df_ = entity_df.copy()
entity_df_['언론사ID'] = entity_df_['doc_id'].map(press_map)

# 1. 전체 개체명·분류별 출현빈도
entity_frequency_df = (
    entity_df_
    .groupby(['개체명', '분류'], as_index=False)
    .size()
    .rename(columns={'size': '출현빈도'})
    .sort_values(
        ['출현빈도', '개체명', '분류'],
        ascending=[False, True, True],
        kind='stable',
    )
) 

entity_frequency_df.to_excel(OUTPUT_DIR / '개체명 목록_빈도.xlsx', index=False)
entity_frequency_df.to_csv(OUTPUT_DIR / '개체명 목록_빈도.csv', index=False, encoding='utf-8-sig')

- 컬럼명: 언론사, 개체명, 분류, 파일ID, 문서ID, 문단ID, 문장ID, 출현빈도
- 출현빈도 집계 기준: 개체명, 분류가 문장에 언급된 총 갯수
- 저장 파일명: 개체명 목록_출현위치_(언론사ID).xlsx

In [ ]:
# 마침표로 구분된 식별자를 각 구간의 숫자값 기준으로 정렬
def _dotted_id_sort_key(value):
    parts = str(value).split('.')
    return tuple((0, int(part)) if part.isdigit() else (1, part) for part in parts)

# 2. 언론사별 개체명 출현 위치 및 출현빈도
location_columns = [
    '개체명',
    '분류',
    'file_id',
    'doc_id',
    'paragraph_id',
    'sentence_id',
]

for press_id, press_df in entity_df_.groupby('언론사ID'):
    location_df = (
        press_df
        .groupby(location_columns, as_index=False)
        .size()
        .sort_values(
            ['doc_id', 'paragraph_id', 'sentence_id'],
            key=lambda column: column.map(_dotted_id_sort_key),
            kind='stable',
        )
        .rename(columns={
            'doc_id': '문서ID',
            'file_id': '파일ID',
            'paragraph_id': '문단ID',
            'sentence_id': '문장ID',
            'size': '출현빈도',
        })
    )

    location_df.to_excel(OUTPUT_DIR / f'개체명 목록_출현위치_({press_id}).xlsx', index=False)
    location_df.to_csv(OUTPUT_DIR / f'개체명 목록_출현위치_({press_id}).csv', index=False, encoding='utf-8-sig')

---

- 2개 산출물 출현 빈도수 검증

In [1]:
import pandas as pd

def read_excel_safe(path):
    return pd.read_excel(path, engine="calamine")

In [4]:
from pathlib import Path
import pandas as pd

XLSX_OUTPUT_DIR = Path(r'C:\work\ner\outputs\20261124_final\개체명목록_기사190건_xlsx')
frequency_path = XLSX_OUTPUT_DIR / '개체명 목록_빈도.xlsx'
location_paths = sorted(XLSX_OUTPUT_DIR.glob('개체명 목록_출현위치_(*).xlsx'))

frequency_columns = ['개체명', '분류', '출현빈도']
location_columns = ['개체명', '분류', '파일ID', '문서ID', '문단ID', '문장ID', '출현빈도']

In [ ]:
frequency_df = read_excel_safe(frequency_path)
assert list(frequency_df.columns) == frequency_columns, (f'전체 빈도 XLSX 컬럼 불일치: {list(frequency_df.columns)}')

frequency_total = frequency_df['출현빈도'].sum()
location_total = 0

for location_path in location_paths:
    location_df = read_excel_safe(location_path)
    assert list(location_df.columns) == location_columns, (f'출현위치 XLSX 컬럼 불일치 ({location_path.name}): {list(location_df.columns)}')
    location_total += location_df['출현빈도'].sum()

assert location_total == frequency_total, (f'출현빈도 합계 불일치: 전체={frequency_total}, 언론사별={location_total}')

In [8]:
location_total

np.int64(6058)

In [9]:
frequency_total

np.int64(6057)